# Flickr30k Report Package

Đóng gói bảng, báo cáo, kết quả khóa và ảnh định tính thành một ZIP để
viết khóa luận/bài báo. Notebook không đọc lại tập Test.

## Input bắt buộc — đúng 3 output

1. `F30K_T3S3_HeldOut_Test_OneTime`;
2. `F30K_Paper_Tables`;
3. `F30K_Final_Closeout`.

CPU, Accelerator None, Internet Off; dự kiến dưới 1 phút.


## Cell 1 — Đóng gói hồ sơ báo cáo có manifest


In [1]:
import hashlib
import json
import os
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
EXPERIMENT_ID = "F30K_Report_Package"
WORK_ROOT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT", f"/kaggle/working/{EXPERIMENT_ID}"
))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def unique_named(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (f"Cần đúng một {name}", matches)
    return matches[0]

def load_json(name):
    path = unique_named(name)
    return path, json.loads(path.read_text(encoding="utf-8"))

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

def verify_manifest(root):
    manifest_path = root / "MANIFEST.json"
    assert manifest_path.is_file(), manifest_path
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    rows = manifest["artifacts"]
    for row in rows:
        path = root / row["relative_path"]
        assert path.is_file(), path
        assert path.stat().st_size == int(row["size_bytes"]), path
        assert sha256(path) == row["sha256"], path
    return manifest_path, manifest


heldout_path, heldout = load_json(
    "F30K_T3S3_HeldOut_Test_OneTime_RESULT.json"
)
tables_path, paper_tables = load_json("F30K_Paper_Tables_RESULT.json")
closeout_path, closeout = load_json("F30K_Final_Closeout_RESULT.json")

for result_path in [heldout_path, tables_path, closeout_path]:
    verify_manifest(result_path.parent)

assert heldout["held_out_test_authorization_remaining"] == 0
assert paper_tables["held_out_test_reopened"] is False
assert closeout["status"] == "scientific_evaluation_complete"

PACKAGE_DIR = WORK_ROOT / "F30K_REPORT_PACKAGE"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

source_files = [
    closeout_path,
    closeout_path.parent / "F30K_FINAL_CLOSEOUT.md",
    tables_path,
    tables_path.parent / "F30K_PAPER_TABLES.md",
    tables_path.parent / "table_1_validation_methods_x100.csv",
    tables_path.parent / "table_2_heldout_test_x100.csv",
    tables_path.parent / "table_3_deployment.csv",
    heldout_path,
    heldout_path.parent / "heldout_test_metrics_x100.csv",
    heldout_path.parent / "heldout_test_qualitative_3_images.png",
]
assert all(path.is_file() for path in source_files), source_files

copied = []
for source in source_files:
    destination = PACKAGE_DIR / source.name
    shutil.copy2(source, destination)
    copied.append(destination)

index_path = PACKAGE_DIR / "README.md"
index_path.write_text(
    f"""# Flickr30k report package

## Kết quả chính

- Phương pháp: T3→S3 Direct Logit-KD, seed 42.
- Validation ba seed: Δ CIDEr-D
  **{closeout['selected_method']['validation_three_seed_delta_CIDEr_D_x100']:+.2f}**.
- Held-out Test CIDEr-D: **{closeout['held_out_test_metrics_x100']['CIDEr_D']:.2f}**.
- Gói hỗn hợp: **{closeout['deployment']['selected_MiB']:.2f} MiB**.
- Giảm dung lượng: **{closeout['deployment']['size_reduction_percent']:.2f}%**.
- Held-out Test đã mở đúng một lần; không được dùng lại để thay đổi mô hình.

Các metric BLEU, METEOR, ROUGE-L và CIDEr-D trong bảng đã nhân 100 và
làm tròn hai chữ số. CE giữ nguyên thang gốc.
""",
    encoding="utf-8",
)
copied.append(index_path)

package_manifest = {
    "schema_version": "f30k_report_package_manifest_v1",
    "created_at_utc": now_utc(),
    "artifact_count": len(copied),
    "held_out_test_reopened": False,
    "artifacts": [
        {
            "relative_path": path.relative_to(PACKAGE_DIR).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in copied
    ],
}
package_manifest_path = PACKAGE_DIR / "MANIFEST.json"
write_json(package_manifest_path, package_manifest)

zip_path = WORK_ROOT / "F30K_REPORT_PACKAGE.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PACKAGE_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(PACKAGE_DIR.parent))

result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
result = {
    "schema_version": "f30k_report_package_v1",
    "created_at_utc": now_utc(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "package_artifact_count": len(copied),
    "package_zip_bytes": zip_path.stat().st_size,
    "package_zip_sha256": sha256(zip_path),
    "held_out_test_reopened": False,
    "scientific_values_changed": False,
}
write_json(result_path, result)

manifest_path = WORK_ROOT / "MANIFEST.json"
artifacts = [result_path, zip_path, package_manifest_path]
write_json(manifest_path, {
    "schema_version": "f30k_report_package_output_manifest_v1",
    "created_at_utc": now_utc(),
    "artifact_count": len(artifacts),
    "artifacts": [
        {
            "relative_path": path.relative_to(WORK_ROOT).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in artifacts
    ],
})
verify_manifest(WORK_ROOT)

print("Report package         : COMPLETE")
print("Files copied           :", len(copied))
print("Held-out Test reopened : False")
print("ZIP                     :", zip_path)
print("ZIP SHA-256             :", sha256(zip_path))
print("Manifest                :", manifest_path)


Report package         : COMPLETE
Files copied           : 11
Held-out Test reopened : False
ZIP                     : /kaggle/working/F30K_Report_Package/F30K_REPORT_PACKAGE.zip
ZIP SHA-256             : 8592735058d758579671dddac44d669bf697e8045a2bd8c00de6e26d9b26ad64
Manifest                : /kaggle/working/F30K_Report_Package/MANIFEST.json
